# DARA — Dual-Spine AI Research Architecture
## Colab integration bootstrap · v1.0 · 14 October 2026

**GPT/ChatGPT** coordinates. **Google Drive** is the Knowledge & Data Spine. **GitHub** is the Technology & Code Spine. **Google Colab + Gemini + Gemini CLI (Cloud Shell) + Google Cloud** form the execution ecosystem.

- **Google Cloud project:** `third-serenity-428705-g2`
- **Canonical architecture folder:** [DARA_ResearchOS (Google Drive)](https://drive.google.com/drive/folders/1OXZ6v1zB0G7wmlGezLCAvm8I4Egr2UPL)
- **Versioned architecture:** [GitHub DARA](https://github.com/Saehon/Saeid-Homayoun/tree/main/DARA)
- **Notebook source:** [GitHub](https://github.com/Saehon/Saeid-Homayoun/blob/main/DARA/colab/DARA_Integration_Bootstrap.ipynb)

**Read before execution:** This notebook does not grant GPT remote control over Colab. Authorize Google Drive and Google Cloud separately. It creates non-destructive test files only inside `MyDrive/DARA_ResearchOS/`; no unrelated files are edited. It does **not** enable Google APIs, run chargeable model inference, change IAM/billing, push to GitHub, or require `OPENAI_API_KEY`.


### 1 — Verify Colab runtime and load DARA configuration


In [ ]:
import os, json, sys, subprocess, platform, uuid
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

PROJECT_ID = "third-serenity-428705-g2"
REPO = "Saehon/Saeid-Homayoun"
ARCHITECTURE_PATH = "DARA/ARCHITECTURE.md"
DRIVE_ROOT = Path("/content/drive/MyDrive")
DARA_DIR = DRIVE_ROOT / "DARA_ResearchOS"
GITHUB_BASE = "https://github.com/" + REPO
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
checks = {
    "Colab Python runtime": "PASS" if "google.colab" in sys.modules else "NOT VERIFIED",
    "GitHub notebook code bridge": "NOT TESTED",
    "Google Drive read/write": "NOT TESTED",
    "Google Cloud authorization": "NOT TESTED",
    "Gemini in Colab UI": "MANUAL VERIFICATION REQUIRED",
    "Gemini CLI in Cloud Shell": "MANUAL VERIFICATION REQUIRED",
    "Vertex AI / BigQuery API access": "NOT TESTED (not enabled or called)",
    "OpenAI API from Colab": "NOT REQUIRED (optional)"
}
print("DARA run:", RUN_ID, "| Python:", platform.python_version())
print("Configured Google Cloud project:", PROJECT_ID)
print("Local runtime:", sys.executable)


### 2 — Read-only check: GitHub code bridge

This tests whether public DARA architecture code is available from GitHub. It does **not** create a GitHub login or write to any repository.


In [ ]:
api = "https://api.github.com/repos/" + REPO + "/contents/" + ARCHITECTURE_PATH
try:
    request = Request(api, headers={"User-Agent": "DARA-Colab-Integration-Check"})
    with urlopen(request, timeout=20) as result:
        assert result.status == 200
        github_metadata = json.load(result)
    assert github_metadata.get("path") == ARCHITECTURE_PATH
    checks["GitHub notebook code bridge"] = "PASS (DARA architecture accessible; no GitHub writes)"
    print("GitHub bridge PASS:", github_metadata.get("html_url"))
except Exception as exc:
    checks["GitHub notebook code bridge"] = "PARTIAL (could not verify API: " + type(exc).__name__ + ")"
    print("Public GitHub availability check not verified:", type(exc).__name__)


### 3 — Mount Google Drive

Google will request authorization in the browser. Do not share authentication codes, tokens or secrets in notebook cells or chat.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Drive mount failed: /content/drive/MyDrive missing")
DARA_DIR.mkdir(parents=True, exist_ok=True)
print("Drive mounted. DARA folder available:", DARA_DIR)


### 4 — Verify Google Drive write/readback without overwriting anything

Only a uniquely named text file is created under `DARA_ResearchOS/`. It is preserved for audit evidence.


In [ ]:
test_file = DARA_DIR / ("DARA_DRIVE_READBACK_" + RUN_ID + ".txt")
marker = "DARA Drive verification; Run ID: " + RUN_ID
with test_file.open("x", encoding="utf-8") as f:
    f.write(marker)
if test_file.read_text(encoding="utf-8") != marker:
    raise RuntimeError("Drive readback mismatch")
checks["Google Drive read/write"] = "PASS (new file saved and read back)"
print("Drive readback PASS:", test_file.name)


### 5 — Authenticate Google Cloud and select the precise project

Authorize the Google Cloud account when prompted. This tests **credentials and gcloud project configuration only**; it does not prove BigQuery or Vertex AI are enabled.


In [ ]:
from google.colab import auth
import google.auth

auth.authenticate_user()
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
               check=True, capture_output=True, text=True)
actual = subprocess.run(["gcloud", "config", "get-value", "project"],
                        check=True, capture_output=True, text=True).stdout.strip()
adc, reported_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"])
assert actual == PROJECT_ID
assert adc is not None
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
checks["Google Cloud authorization"] = "PASS (credentials available, project matches; APIs not tested)"
print("Google Cloud configuration PASS:", actual)
print("ADC project metadata:", reported_project)


### 6 — Gemini and Gemini CLI: verification is intentionally separate

**Gemini in Colab** is a built-in assistant: open its sidebar and ask `Explain the DARA dual-spine architecture in one sentence`. The Python notebook cannot verify that sidebar interaction.

**Gemini CLI** runs in **Google Cloud Shell**, a different terminal and identity/session context. Open [Cloud Shell](https://console.cloud.google.com/) and manually run:

```bash
gcloud config set project third-serenity-428705-g2
gcloud config get-value project
gemini --version
```

`gemini --version` only confirms installation. Running `gemini` may require interactive authentication and use quota. Neither is automatically marked PASS here.

**Gemini via Vertex AI, BigQuery and OpenAI API** are optional and are deliberately not invoked; separate IAM/API/billing and approval checks are required.


### 7 — Save verified setup report to Google Drive

The report includes no API keys or identity credentials. The report is created only if Drive readback was successful. Existing files are never deleted or overwritten.


In [ ]:
report = {
    "system": "DARA — Dual-Spine AI Research Architecture",
    "version": "1.0",
    "run_id": RUN_ID,
    "utc_generated": datetime.now(timezone.utc).isoformat(),
    "github_repository": REPO,
    "google_cloud_project": PROJECT_ID,
    "python_version": platform.python_version(),
    "checks": checks,
    "interpretation": (
        "PASS means only the explicitly executed test passed. "
        "No Gemini/GPT model inference, paid cloud API or autonomous Colab control was performed."
    )
}
if not checks["Google Drive read/write"].startswith("PASS"):
    raise RuntimeError("Refusing to save: Drive has not passed readback verification.")
report_file = DARA_DIR / ("DARA_COLAB_INTEGRATION_REPORT_" + RUN_ID + ".json")
with report_file.open("x", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
readback = json.loads(report_file.read_text(encoding="utf-8"))
assert readback["run_id"] == RUN_ID and readback["google_cloud_project"] == PROJECT_ID
print("Verified report saved in Google Drive:", report_file)
print(json.dumps(checks, indent=2))


## After setup: verified DARA workflow

1. GPT reviews/creates approved code through its connected GitHub app.
2. GitHub preserves scripts, notebooks, commits and PR approvals.
3. You open the notebook from GitHub in Colab, authorize Drive/Cloud, and execute it.
4. Colab writes validated outputs and non-sensitive audit logs to the **existing canonical Drive research folder** for that project.
5. GPT examines saved Drive outputs through its **independently connected** Google Drive app.
6. Any changes to methodology, data access or cloud spending require a separate decision.

**Do not move existing research folders.** DARA is an architecture and master-index layer, not a wholesale project migration. This notebook establishes only connectivity checks; it does not start any CAM, ICFR or valuation study.
